# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

In [3]:
%pip install "langchain-elasticsearch<0.3"

  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
     ---------------------------------------- 0.0/71.9 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/71.9 kB ? eta -:--:--
     ---------------------------------- ----- 61.4/71.9 kB 1.1 MB/s eta 0:00:01
     -------------------------------------- 71.9/71.9 kB 656.1 kB/s eta 0:00:00
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   --- ------------------------------------ 71.7/952.8 kB 3.8 MB/s eta 0:00:01
   ------- -------------------------------- 174.1/952.8 kB 2.6 MB/s eta 0:00:01
   --------- ------------------------------ 235.5/952.8 kB 2.0 MB/s eta 0:00:01
   ------------- -------------------------- 3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.7.3 requires langchain-core<2.0.0,>=1.7.0, but you have langchain-core 0.2.43 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install --upgrade "langchain==0.2.17" "langchain-core==0.2.43" "langchain-community==0.2.19" "langchain-openai==0.1.25" "langchain-elasticsearch==0.2.2" "jq==1.12.0"

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_community-0.2.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl.metadata (2.6 kB)
  Using cached sqlalchemy-2.1.6-cp311-cp311-win_amd64.whl.metadata (9.9 kB)
  Using cached aiohttp-3.14.5-cp311-cp311-win_amd64.whl.metadata (8.5 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached multidict-7.2.1-cp311-cp311-win_amd64.whl.metadata (

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [2]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-docs-rag",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [3]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [5]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [9]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-docs-rag",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

In [10]:
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

llm = ChatOpenAI(
    model="gpt-6-luna",
    temperature=1,
    openai_api_key=OPENAI_API_KEY,
    model_kwargs={"reasoning_effort": "none"}
)

retriever = MultiQueryRetriever.from_llm(
    vectorstore.as_retriever(),
    llm
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [12]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do?', 'Who is part of the sales team responsible for NASA accounts?', 'How is the sales team for NASA organized?']


---- Answer ----
content='NASA refers to the North America–South America sales region, covering the Americas. It has two Area Vice-Presidents: Laura Martinez leads North America, and Gary Johnson leads South America.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 1543, 'total_tokens': 1584, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 1540}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-6b4b44d4-8580-4b15-bb3f-c023f93ec778-0' usage_metadata={'input_tokens': 1543, 'output_tokens': 41, 'total_tokens': 1584}


<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [16]:
import logging

logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [17]:
ans = chain.invoke(
    "What are the main rules for employees working from home?"
)

print("---- Answer 1 ----")
print(ans)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain.retrievers.multi_query:Generated queries: ['What are the key remote-work policies and requirements employees must follow?', 'What rules and expectations apply to employees who work from home?', 'What are employees’ responsibilities and restrictions when working remotely?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://cf43ab92ac3b468cbed8d12688d68a6f.es.europe-west3.gcp.elastic.cloud:443/workplace-docs-rag/_search?_source_includes=metadata,text [status:200 duration:0.060s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://cf43ab92ac3b468cbed8d12688d68a6f.es.europe-west3.gcp.elastic.cloud:443/workplace-docs-rag/_search?_source_includes=metadata,text [status:200 duration:0.064s]
INFO:httpx:HTTP Request: POST https://api.

---- Answer 1 ----
content='Employees working from home should:\n\n- Have approval from their supervisor and HR, and be in a role that can be performed remotely.\n- Maintain a safe, comfortable, and productive workspace.\n- Keep regular contact with their supervisor and colleagues using approved communication tools.\n- Work their regular hours and be available during normal business hours, unless they have agreed otherwise with their supervisor. Communicate changes in availability to their supervisor and HR.\n- Meet the same performance and productivity expectations as in the office.\n- Accurately record work hours; non-exempt employees need supervisor approval before working overtime.\n- Protect company equipment, confidential information, and data, and report security breaches to IT.\n- Take regular breaks and maintain a healthy work-life balance.\n\nThe policy also notes that, starting May 1, 2023, employees are expected to work in the office three days a week and remotely two days 

In [18]:
ans = chain.invoke(
    "What is the company's vacation policy, and how should employees request time off?"
)

print("---- Answer 2 ----")
print(ans)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain.retrievers.multi_query:Generated queries: ['What vacation and paid-time-off benefits does the company offer, and what rules apply to taking leave?', 'How should employees submit a vacation or time-off request, and how much advance notice or approval is required?', 'What is the company’s process for requesting and getting approval for holidays, vacation days, or other time off?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://cf43ab92ac3b468cbed8d12688d68a6f.es.europe-west3.gcp.elastic.cloud:443/workplace-docs-rag/_search?_source_includes=metadata,text [status:200 duration:0.128s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://cf43ab92ac3b468cbed8d12688d68a6f.es.europe-west3.gcp.elastic.cloud:443/workplace-docs-rag/_s

---- Answer 2 ----
content='The vacation policy is intended to support work-life balance by giving employees time to rest and recharge. It applies to full-time and part-time employees who have completed their probationary period.\n\n- **Accrual:** Full-time employees accrue vacation at a rate of **[X hours] per month ([Y days] per year)**. Part-time employees accrue it pro rata based on scheduled work hours. Accrual starts on the first day of employment, but vacation may be taken only after probation is complete.\n- **Carryover:** Unused vacation can be carried into the next year up to **[Z days]**; any excess is forfeited.\n- **Requesting time off:** Submit a request to your supervisor at least **[A weeks] in advance**, stating the start and end dates. The supervisor reviews requests based on business needs and coverage. Employees are encouraged to avoid peak periods where possible; requests during peak periods may have limits or require more advance notice.\n- **Pay:** Approved vacat

In [ ]:
## Reflection

I think MultiQueryRetriever is useful because it creates different versions of the same question, so Elasticsearch has more chances to find the right documents.

The pipeline worked with my question about the NASA sales team and returned a relevant answer. However, I had a problem with the document metadata, so I removed the document name from the prompt to make the chain work.

I learned that it is important to check the generated queries and the retrieved context, because the final answer is only reliable if the relevant information is found in the documents.